# Notebook 03 - Demo End-to-End

**Tech Challenge - Fase 5 - Hackaton FIAP**
**Modelagem de Ameaças utilizando IA (STRIDE)**

Pipeline completo: Imagem -> Provedor -> Componentes -> STRIDE -> Relatório

Executa para as 2 arquiteturas de teste (AWS e Azure) e apresenta:
- Visualização com bounding boxes
- Resumo do relatório STRIDE
- Métricas de execução

**GPU necessária**: T4

## 1. Setup

In [ ]:
!pip install -q transformers anthropic supervision fpdf2 timm einops flash_attn

In [ ]:
import os
import io
import json
import base64
import time
from pathlib import Path
from datetime import datetime
from collections import Counter

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import supervision as sv
from transformers import AutoProcessor, AutoModelForCausalLM
import anthropic
from fpdf import FPDF

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/hackaton-stride')
IMAGES_DIR = DRIVE_BASE / 'test_images'
OUTPUT_DIR = DRIVE_BASE / 'outputs'
DETECTIONS_DIR = OUTPUT_DIR / 'detections'
REPORTS_DIR = OUTPUT_DIR / 'reports'

for d in [DETECTIONS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

## 2. Carregar Florence-2

In [ ]:
FLORENCE_MODEL_ID = 'microsoft/Florence-2-large'

print("Carregando Florence-2...")
florence_processor = AutoProcessor.from_pretrained(
    FLORENCE_MODEL_ID, trust_remote_code=True
)
florence_model = AutoModelForCausalLM.from_pretrained(
    FLORENCE_MODEL_ID, trust_remote_code=True,
    torch_dtype=torch.float16
).to('cuda')
print("Florence-2 pronto!")

## 3. Definir Pipeline Completo

In [ ]:
# Classes de detecção
COMPONENT_CLASSES = {
    'waf_firewall': ['AWS WAF', 'AWS Shield', 'Azure Firewall'],
    'cdn': ['Amazon CloudFront', 'Azure CDN'],
    'load_balancer': ['ALB', 'NLB', 'Azure Load Balancer'],
    'vpc_vnet': ['VPC', 'VNet', 'Subnet'],
    'compute': ['EC2', 'SEI/SIP', 'App Service', 'VM'],
    'auto_scaling': ['Auto Scaling Group', 'VMSS'],
    'orchestrator': ['Logic Apps', 'Step Functions', 'Lambda'],
    'database': ['RDS', 'Aurora', 'Azure SQL', 'Cosmos DB'],
    'cache': ['ElastiCache', 'Azure Cache for Redis'],
    'storage': ['S3', 'EFS', 'Azure Blob', 'NFS'],
    'api_gateway': ['API Gateway', 'Azure API Management'],
    'developer_portal': ['Developer Portal'],
    'web_service': ['REST', 'SOAP', 'SaaS Service'],
    'auth_identity': ['IAM', 'Microsoft Entra', 'Cognito'],
    'kms_encryption': ['AWS KMS', 'Azure Key Vault'],
    'monitoring': ['CloudWatch', 'CloudTrail', 'Azure Monitor'],
    'backup': ['AWS Backup', 'Azure Backup'],
    'messaging': ['SES', 'SQS', 'SNS', 'Azure Service Bus'],
    'user_actor': ['Usuário', 'Developer', 'Client'],
}

ALL_SERVICES = []
for cls, services in COMPONENT_CLASSES.items():
    for svc in services:
        ALL_SERVICES.append(f'{svc} ({cls})')
SERVICES_LIST = ', '.join(ALL_SERVICES)

In [ ]:
# === Funções auxiliares ===

def encode_image_base64(image_path: str) -> str:
    with open(image_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def encode_pil_image_base64(pil_image: Image.Image) -> str:
    buffer = io.BytesIO()
    pil_image.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('utf-8')


# === Passo 1: Identificar provedor ===

def identify_provider(image_path: str) -> str:
    image_b64 = encode_image_base64(image_path)
    response = client.messages.create(
        model='claude-sonnet-4-6-20250514',
        max_tokens=100,
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': image_b64}},
                {'type': 'text', 'text': 'Analise esta imagem de diagrama de arquitetura de software. Identifique o provedor cloud principal. Responda APENAS: "AWS" ou "Azure".'}
            ]
        }]
    )
    text = response.content[0].text.strip().upper()
    return 'AWS' if 'AWS' in text else 'Azure' if 'AZURE' in text else 'AWS'


# === Passo 2a: Detecção com Florence-2 ===

def detect_florence(image: Image.Image) -> dict:
    all_boxes, all_labels = [], []
    for task in ['<OD>', '<DENSE_REGION_CAPTION>', '<REGION_PROPOSAL>']:
        inputs = florence_processor(text=task, images=image, return_tensors='pt').to('cuda', torch.float16)
        with torch.no_grad():
            ids = florence_model.generate(input_ids=inputs['input_ids'], pixel_values=inputs['pixel_values'], max_new_tokens=2048, num_beams=3)
        text = florence_processor.batch_decode(ids, skip_special_tokens=False)[0]
        result = florence_processor.post_process_generation(text, task=task, image_size=(image.width, image.height))
        if task in result:
            data = result[task]
            all_boxes.extend(data.get('bboxes', []))
            labels = data.get('labels', ['region'] * len(data.get('bboxes', [])))
            all_labels.extend(labels)
    return {'bboxes': all_boxes, 'labels': all_labels}


def filter_boxes(detections: dict, image_size: tuple) -> dict:
    img_area = image_size[0] * image_size[1]
    boxes, labels = [], []
    for bbox, label in zip(detections['bboxes'], detections['labels']):
        x1, y1, x2, y2 = bbox
        w, h = x2 - x1, y2 - y1
        ratio = (w * h) / img_area
        if 0.001 <= ratio <= 0.5 and w > 10 and h > 10:
            boxes.append([x1, y1, x2, y2])
            labels.append(label)

    if not boxes:
        return {'bboxes': [], 'labels': []}

    # NMS
    boxes_np = np.array(boxes)
    keep, used = [], set()
    for i in range(len(boxes_np)):
        if i in used: continue
        keep.append(i)
        for j in range(i+1, len(boxes_np)):
            if j in used: continue
            xi1 = max(boxes_np[i][0], boxes_np[j][0])
            yi1 = max(boxes_np[i][1], boxes_np[j][1])
            xi2 = min(boxes_np[i][2], boxes_np[j][2])
            yi2 = min(boxes_np[i][3], boxes_np[j][3])
            inter = max(0, xi2-xi1) * max(0, yi2-yi1)
            a_i = (boxes_np[i][2]-boxes_np[i][0]) * (boxes_np[i][3]-boxes_np[i][1])
            a_j = (boxes_np[j][2]-boxes_np[j][0]) * (boxes_np[j][3]-boxes_np[j][1])
            if inter / (a_i + a_j - inter + 1e-6) > 0.5:
                used.add(j)

    return {'bboxes': [boxes[i] for i in keep], 'labels': [labels[i] for i in keep]}


# === Passo 2b: Classificação com Claude Vision ===

def classify_crop(crop: Image.Image, provider: str, full_b64: str) -> dict:
    crop_b64 = encode_pil_image_base64(crop)
    prompt = f"""Analise este recorte de um diagrama de arquitetura {provider}.
A segunda imagem mostra o diagrama completo para contexto.
Identifique o componente/serviço cloud.
Serviços possíveis: {SERVICES_LIST}
Responda APENAS em JSON: {{"name": "Nome do serviço", "class": "classe"}}
Se não identificável: {{"name": "unknown", "class": "unknown"}}"""

    response = client.messages.create(
        model='claude-sonnet-4-6-20250514', max_tokens=200,
        messages=[{'role': 'user', 'content': [
            {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': crop_b64}},
            {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': full_b64}},
            {'type': 'text', 'text': prompt}
        ]}]
    )
    text = response.content[0].text.strip()
    try:
        s, e = text.find('{'), text.rfind('}') + 1
        if s >= 0 and e > s:
            return json.loads(text[s:e])
    except json.JSONDecodeError:
        pass
    return {'name': 'unknown', 'class': 'unknown'}


# === Passo 3: Análise STRIDE ===

def analyze_stride(component: dict, all_components: list, provider: str) -> dict:
    arch_context = "\n".join(f"  - {c['name']} ({c['class']})" for c in all_components)
    prompt = f"""Você é um especialista em segurança de software.

Analise o componente usando STRIDE. Arquitetura {provider}.

Componente: {component['name']} ({component['class']})

Outros componentes:
{arch_context}

Para cada categoria STRIDE, forneça: threat, risk (Alto/Médio/Baixo), countermeasure, reference.
Contramedidas específicas de {provider}. Referencie {"AWS Well-Architected" if provider == "AWS" else "Azure Security Benchmark"}.

Responda APENAS em JSON:
{{"spoofing": {{"threat": "...", "risk": "...", "countermeasure": "...", "reference": "..."}},
"tampering": {{...}}, "repudiation": {{...}}, "information_disclosure": {{...}},
"denial_of_service": {{...}}, "elevation_of_privilege": {{...}}}}"""

    response = client.messages.create(
        model='claude-sonnet-4-6-20250514', max_tokens=2000,
        messages=[{'role': 'user', 'content': prompt}]
    )
    text = response.content[0].text.strip()
    try:
        s, e = text.find('{'), text.rfind('}') + 1
        if s >= 0 and e > s:
            return json.loads(text[s:e])
    except json.JSONDecodeError:
        pass
    return {}

In [ ]:
# === Passo 4: Geração de relatório PDF ===

def generate_summary(stride_results: list) -> dict:
    total_threats = 0
    risk_counts = {'Alto': 0, 'Médio': 0, 'Baixo': 0}
    high_risk = []
    for r in stride_results:
        h = 0
        for cat, d in r.get('threats', {}).items():
            if isinstance(d, dict):
                total_threats += 1
                risk = d.get('risk', 'Baixo')
                if risk in risk_counts: risk_counts[risk] += 1
                if risk == 'Alto': h += 1
        if h > 0:
            high_risk.append({'name': r['name'], 'class': r['class'], 'high_risk_count': h})
    return {
        'total_components': len(stride_results),
        'total_threats': total_threats,
        'risk_distribution': risk_counts,
        'high_risk_components': sorted(high_risk, key=lambda x: x['high_risk_count'], reverse=True)
    }


def generate_pdf(stride_results: list, summary: dict, provider: str, arch_index: int, output_path: str):
    pdf = FPDF()
    pdf.add_font('DejaVu', '', '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', uni=True)
    pdf.add_font('DejaVu', 'B', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', uni=True)
    pdf.alias_nb_pages()

    # Capa
    pdf.add_page()
    pdf.ln(60)
    pdf.set_font('DejaVu', 'B', 28)
    pdf.cell(0, 15, 'Relatório STRIDE', 0, 1, 'C')
    pdf.ln(10)
    pdf.set_font('DejaVu', '', 16)
    pdf.cell(0, 10, f'Arquitetura {arch_index} - {provider}', 0, 1, 'C')
    pdf.ln(20)
    pdf.set_font('DejaVu', '', 12)
    pdf.cell(0, 8, 'Tech Challenge - Fase 5 - Hackaton FIAP', 0, 1, 'C')
    pdf.cell(0, 8, f'Gerado em: {datetime.now().strftime("%d/%m/%Y %H:%M")}', 0, 1, 'C')

    # Resumo
    pdf.add_page()
    pdf.set_font('DejaVu', 'B', 18)
    pdf.cell(0, 12, 'Resumo Executivo', 0, 1)
    pdf.ln(5)
    pdf.set_font('DejaVu', '', 11)
    pdf.cell(0, 7, f"Componentes: {summary['total_components']}", 0, 1)
    pdf.cell(0, 7, f"Ameaças: {summary['total_threats']}", 0, 1)
    pdf.cell(0, 7, f"Alto: {summary['risk_distribution']['Alto']} | Médio: {summary['risk_distribution']['Médio']} | Baixo: {summary['risk_distribution']['Baixo']}", 0, 1)
    pdf.ln(5)
    if summary['high_risk_components']:
        pdf.set_font('DejaVu', 'B', 12)
        pdf.cell(0, 8, 'Componentes Críticos:', 0, 1)
        pdf.set_font('DejaVu', '', 10)
        for c in summary['high_risk_components']:
            pdf.cell(0, 6, f"  - {c['name']}: {c['high_risk_count']} riscos altos", 0, 1)

    # Tabela de componentes
    pdf.add_page()
    pdf.set_font('DejaVu', 'B', 18)
    pdf.cell(0, 12, 'Componentes Detectados', 0, 1)
    pdf.ln(5)
    pdf.set_font('DejaVu', 'B', 9)
    pdf.set_fill_color(50, 50, 50)
    pdf.set_text_color(255, 255, 255)
    pdf.cell(10, 8, '#', 1, 0, 'C', fill=True)
    pdf.cell(70, 8, 'Componente', 1, 0, 'C', fill=True)
    pdf.cell(50, 8, 'Classe', 1, 0, 'C', fill=True)
    pdf.cell(30, 8, 'Provedor', 1, 1, 'C', fill=True)
    pdf.set_font('DejaVu', '', 8)
    pdf.set_text_color(30, 30, 30)
    for i, r in enumerate(stride_results):
        fill = i % 2 == 0
        if fill: pdf.set_fill_color(245, 245, 245)
        pdf.cell(10, 7, str(i+1), 1, 0, 'C', fill=fill)
        pdf.cell(70, 7, r['name'][:35], 1, 0, 'L', fill=fill)
        pdf.cell(50, 7, r['class'], 1, 0, 'C', fill=fill)
        pdf.cell(30, 7, r['provider'], 1, 1, 'C', fill=fill)

    # Detalhes STRIDE por componente
    cat_names = {
        'spoofing': 'S - Spoofing', 'tampering': 'T - Tampering',
        'repudiation': 'R - Repudiation', 'information_disclosure': 'I - Info Disclosure',
        'denial_of_service': 'D - Denial of Service', 'elevation_of_privilege': 'E - Elevation'
    }
    risk_colors = {'Alto': (220, 53, 69), 'Médio': (255, 193, 7), 'Baixo': (40, 167, 69)}

    for result in stride_results:
        pdf.add_page()
        pdf.set_font('DejaVu', 'B', 14)
        pdf.set_text_color(30, 30, 30)
        pdf.cell(0, 10, result['name'], 0, 1)
        pdf.set_font('DejaVu', '', 10)
        pdf.set_text_color(100, 100, 100)
        pdf.cell(0, 6, f"Classe: {result['class']} | Provedor: {result['provider']}", 0, 1)
        pdf.ln(3)
        for cat_key, cat_name in cat_names.items():
            threats = result.get('threats', {})
            if cat_key not in threats or not isinstance(threats[cat_key], dict):
                continue
            d = threats[cat_key]
            if pdf.get_y() > 240: pdf.add_page()
            pdf.set_font('DejaVu', 'B', 10)
            pdf.set_text_color(30, 30, 30)
            risk = d.get('risk', 'Baixo')
            r, g, b = risk_colors.get(risk, (100, 100, 100))
            pdf.cell(140, 7, cat_name, 0, 0)
            pdf.set_fill_color(r, g, b)
            pdf.set_text_color(255, 255, 255)
            pdf.cell(25, 7, f' {risk} ', 0, 1, 'C', fill=True)
            pdf.set_text_color(30, 30, 30)
            pdf.set_font('DejaVu', '', 9)
            pdf.multi_cell(0, 5, f"Ameaça: {d.get('threat', 'N/A')}")
            pdf.multi_cell(0, 5, f"Contramedida: {d.get('countermeasure', 'N/A')}")
            pdf.set_font('DejaVu', '', 8)
            pdf.set_text_color(100, 100, 100)
            pdf.multi_cell(0, 4, f"Ref: {d.get('reference', 'N/A')}")
            pdf.set_text_color(30, 30, 30)
            pdf.ln(2)

    pdf.output(str(output_path))
    return pdf.page_no()

## 4. Pipeline End-to-End

In [ ]:
def run_pipeline(arch_index: int) -> dict:
    """
    Executa o pipeline completo para uma arquitetura.
    Retorna dict com resultados e métricas.
    """
    print(f"{'='*60}")
    print(f"PIPELINE - Arquitetura {arch_index}")
    print(f"{'='*60}")
    metrics = {}
    t_total = time.time()

    # Carregar imagem
    image_path = IMAGES_DIR / f'arquitetura {arch_index}.png'
    image = Image.open(image_path).convert('RGB')
    print(f"\nImagem: {image_path.name} ({image.size})")

    # --- Passo 1: Identificar provedor ---
    print(f"\n[Passo 1] Identificando provedor cloud...")
    t0 = time.time()
    provider = identify_provider(str(image_path))
    metrics['step1_provider_s'] = round(time.time() - t0, 2)
    print(f"  Provedor: {provider} ({metrics['step1_provider_s']}s)")

    # --- Passo 2a: Detecção Florence-2 ---
    print(f"\n[Passo 2a] Detectando componentes com Florence-2...")
    t0 = time.time()
    raw = detect_florence(image)
    detections = filter_boxes(raw, image.size)
    metrics['step2a_florence_s'] = round(time.time() - t0, 2)
    print(f"  Regiões detectadas: {len(detections['bboxes'])} ({metrics['step2a_florence_s']}s)")

    # --- Passo 2b: Classificação Claude Vision ---
    print(f"\n[Passo 2b] Classificando componentes com Claude Vision...")
    t0 = time.time()
    full_b64 = encode_image_base64(str(image_path))
    components = []
    for idx, bbox in enumerate(detections['bboxes']):
        x1, y1, x2, y2 = [int(c) for c in bbox]
        margin = 10
        crop = image.crop((
            max(0, x1-margin), max(0, y1-margin),
            min(image.width, x2+margin), min(image.height, y2+margin)
        ))
        result = classify_crop(crop, provider, full_b64)
        if result.get('class') != 'unknown':
            components.append({
                'id': idx, 'name': result['name'], 'class': result['class'],
                'bbox': [x1, y1, x2-x1, y2-y1], 'provider': provider
            })
            print(f"  [{idx+1}] {result['name']} ({result['class']})")
        time.sleep(0.5)
    metrics['step2b_classify_s'] = round(time.time() - t0, 2)
    metrics['components_detected'] = len(components)
    print(f"  Componentes: {len(components)} ({metrics['step2b_classify_s']}s)")

    # Salvar detecções
    det_data = {
        'image': image_path.name, 'provider': provider,
        'image_size': {'width': image.width, 'height': image.height},
        'components': components
    }
    det_path = DETECTIONS_DIR / f'componentes_arquitetura_{arch_index}.json'
    with open(det_path, 'w', encoding='utf-8') as f:
        json.dump(det_data, f, ensure_ascii=False, indent=2)

    # --- Passo 3: Análise STRIDE ---
    print(f"\n[Passo 3] Executando análise STRIDE...")
    t0 = time.time()
    stride_results = []
    for comp in components:
        threats = analyze_stride(comp, components, provider)
        stride_results.append({**comp, 'threats': threats})
        risks = [threats[c].get('risk', '?') for c in threats if isinstance(threats[c], dict)]
        high = risks.count('Alto')
        print(f"  {comp['name']}: {high} riscos altos")
        time.sleep(1)
    metrics['step3_stride_s'] = round(time.time() - t0, 2)
    print(f"  Análise concluída ({metrics['step3_stride_s']}s)")

    # --- Passo 4: Relatórios ---
    print(f"\n[Passo 4] Gerando relatórios...")
    t0 = time.time()
    summary = generate_summary(stride_results)

    # JSON
    report = {
        'metadata': {
            'project': 'Tech Challenge - Fase 5 - Hackaton FIAP',
            'methodology': 'STRIDE',
            'generated_at': datetime.now().isoformat(),
            'image': image_path.name, 'provider': provider
        },
        'components': stride_results, 'summary': summary
    }
    json_path = REPORTS_DIR / f'stride_arquitetura_{arch_index}.json'
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    # PDF
    pdf_path = REPORTS_DIR / f'stride_arquitetura_{arch_index}.pdf'
    pages = generate_pdf(stride_results, summary, provider, arch_index, pdf_path)
    metrics['step4_reports_s'] = round(time.time() - t0, 2)
    metrics['total_threats'] = summary['total_threats']
    metrics['risk_distribution'] = summary['risk_distribution']
    print(f"  JSON: {json_path}")
    print(f"  PDF:  {pdf_path} ({pages} páginas)")

    # Visualização com bounding boxes
    if components:
        xyxy = np.array([[c['bbox'][0], c['bbox'][1], c['bbox'][0]+c['bbox'][2], c['bbox'][1]+c['bbox'][3]] for c in components])
        det_sv = sv.Detections(xyxy=xyxy, class_id=np.arange(len(components)))
        labels = [f"{c['name']}" for c in components]
        img_np = np.array(image)
        annotated = sv.BoxAnnotator(thickness=2).annotate(scene=img_np.copy(), detections=det_sv)
        annotated = sv.LabelAnnotator(text_scale=0.4, text_padding=4).annotate(scene=annotated, detections=det_sv, labels=labels)
        ann_path = DETECTIONS_DIR / f'annotated_arquitetura_{arch_index}.png'
        Image.fromarray(annotated).save(ann_path)

    metrics['total_s'] = round(time.time() - t_total, 2)

    return {
        'arch_index': arch_index, 'provider': provider,
        'components': components, 'stride_results': stride_results,
        'summary': summary, 'metrics': metrics,
        'annotated': annotated if components else None
    }

## 5. Executar para Ambas as Arquiteturas

In [ ]:
# Executar pipeline para Arquitetura 1 (AWS)
result_1 = run_pipeline(1)

In [ ]:
# Executar pipeline para Arquitetura 2 (Azure)
result_2 = run_pipeline(2)

## 6. Visualização Comparativa

In [ ]:
# Visualização lado a lado: imagens com bounding boxes
fig, axes = plt.subplots(1, 2, figsize=(24, 12))

for ax, result in zip(axes, [result_1, result_2]):
    if result['annotated'] is not None:
        ax.imshow(result['annotated'])
    ax.set_title(
        f"Arquitetura {result['arch_index']} ({result['provider']})\n"
        f"{result['metrics']['components_detected']} componentes | "
        f"{result['metrics']['total_threats']} ameaças",
        fontsize=14
    )
    ax.axis('off')

plt.suptitle('Detecção de Componentes - Comparativo', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Comparativo de riscos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, result in zip(axes, [result_1, result_2]):
    risks = result['metrics']['risk_distribution']
    colors_bar = ['#dc3545', '#ffc107', '#28a745']
    bars = ax.bar(risks.keys(), risks.values(), color=colors_bar)
    ax.set_title(f"Arq. {result['arch_index']} ({result['provider']})")
    ax.set_ylabel('Quantidade de ameaças')
    for bar, val in zip(bars, risks.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(val), ha='center', fontweight='bold')

plt.suptitle('Distribuição de Riscos STRIDE', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Métricas de Execução

In [ ]:
# Tabela comparativa de métricas
print("=" * 70)
print(f"{'MÉTRICAS DE EXECUÇÃO':^70}")
print("=" * 70)
print(f"{'Métrica':<35} {'Arq. 1 (AWS)':>15} {'Arq. 2 (Azure)':>15}")
print("-" * 70)

m1 = result_1['metrics']
m2 = result_2['metrics']

rows = [
    ('Passo 1 - Provedor (s)', m1['step1_provider_s'], m2['step1_provider_s']),
    ('Passo 2a - Florence-2 (s)', m1['step2a_florence_s'], m2['step2a_florence_s']),
    ('Passo 2b - Classificação (s)', m1['step2b_classify_s'], m2['step2b_classify_s']),
    ('Passo 3 - STRIDE (s)', m1['step3_stride_s'], m2['step3_stride_s']),
    ('Passo 4 - Relatórios (s)', m1['step4_reports_s'], m2['step4_reports_s']),
    ('Tempo total (s)', m1['total_s'], m2['total_s']),
    ('', '', ''),
    ('Componentes detectados', m1['components_detected'], m2['components_detected']),
    ('Total de ameaças', m1['total_threats'], m2['total_threats']),
    ('Riscos altos', m1['risk_distribution']['Alto'], m2['risk_distribution']['Alto']),
    ('Riscos médios', m1['risk_distribution']['Médio'], m2['risk_distribution']['Médio']),
    ('Riscos baixos', m1['risk_distribution']['Baixo'], m2['risk_distribution']['Baixo']),
]

for label, v1, v2 in rows:
    print(f"{label:<35} {str(v1):>15} {str(v2):>15}")

print("=" * 70)
print(f"\nArquivos gerados:")
for idx in [1, 2]:
    print(f"  Arq. {idx}:")
    print(f"    - {DETECTIONS_DIR}/componentes_arquitetura_{idx}.json")
    print(f"    - {DETECTIONS_DIR}/annotated_arquitetura_{idx}.png")
    print(f"    - {REPORTS_DIR}/stride_arquitetura_{idx}.json")
    print(f"    - {REPORTS_DIR}/stride_arquitetura_{idx}.pdf")